In [29]:
import os

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_poisson_deviance

import config
from fixture_loader import load_fixtures, both_prem, filter_prem, add_rest_days
from pipeline.context import load_pipeline_context
from pipeline.player_base import build_player_base, TRANSFERRED_OUT
from pipeline.form import attach_form, build_form, compute_form_score
from pipeline.team_strength import (
    load_elo_all, build_elo_long, attach_match_elo,
    build_team_table, add_diffs, print_team_strength_diagnostics,
)
from pipeline.predictions import build_train_test, fit_pooled_goal_model, predict_fixtures
from pipeline.team_scores import build_team_fixture_scores, attach_team_and_fixture_scores, to_friendly
from pipeline.last_season import attach_last_season_stats
from pipeline.scoring import (
    attach_minutes_factors, attach_expected_return_signals, assemble_board,
    scale_global, expected_fpl_p90, compute_vorp_dynamic,
)
from pipeline.draft_ownership import build_all_boards
from pipeline.board_output import enrich_and_export, print_player_debug

In [ ]:
%load_ext autoreload
%autoreload 2

ctx = load_pipeline_context()

# unpack into globals for ad hoc / exploratory cells
GW, LAST_GW, TOTAL_MINUTES = ctx.GW, ctx.LAST_GW, ctx.TOTAL_MINUTES
GW, LAST_GW = ctx.GW, ctx.LAST_GW
FPL_DATA_FOLDER, PREV_DATA_FOLDER = ctx.FPL_DATA_FOLDER, ctx.PREV_DATA_FOLDER
fixtures_dir, prev_dir = ctx.fixtures_dir, ctx.prev_dir
players, teams, prev_teams = ctx.players, ctx.teams, ctx.prev_teams
playerstats, last_year_stats = ctx.playerstats, ctx.last_year_stats
elo_hist, elo = ctx.elo_hist, ctx.elo

Already up to date.
Next GW: 11
Elo scrape failed (Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.); using last saved snapshot
  -> add this week's Elo: clubelo.com/ENG bookmarklet, then `python get_elo.py`


In [4]:
print(teams[["code","short_name"]].sort_values(by="code"))

    code short_name
15     1        MUN
12     2        LEE
0      3        ARS
16     4        NEW
18     6        TOT
1      7        AVL
5      8        CHE
6      9        COV
8     11        EVE
13    14        LIV
17    17        NFO
7     31        CRY
4     36        BHA
11    40        IPS
14    43        MCI
9     54        FUL
19    56        SUN
10    88        HUL
2     91        BOU
3     94        BRE


In [5]:
#### PLAYER STATS BASE
merged, base = build_player_base(playerstats, players, teams, TRANSFERRED_OUT)

In [6]:
### FORM SCORE
merged = attach_form(merged, GW, FPL_DATA_FOLDER)

In [7]:
### FIXTURE ORGANIZATION
this_seasons_fixtures = load_fixtures(fixtures_dir)
prev_seasons_fixtures = load_fixtures(prev_dir, played_only=True)

## ISOLATING PREM
this_season_prem_matches = filter_prem(this_seasons_fixtures)
prev_fx = filter_prem(both_prem(prev_seasons_fixtures, prev_teams))

print(f"GW{GW} fixtures: {(this_season_prem_matches['gameweek'] == GW).sum()}")

GW11 fixtures: 10


In [8]:
this_season_prem_matches = add_rest_days(this_season_prem_matches, this_seasons_fixtures)
prev_fx = add_rest_days(prev_fx, prev_seasons_fixtures)

print(f"GW{GW} fixtures: {(this_season_prem_matches['gameweek'] == GW).sum()}")

GW11 fixtures: 10


In [9]:
# #### FIXTURE GAMBLER MACHINE LEARNING MODEL
elo_all = load_elo_all()
elo_long = build_elo_long(elo_all)
this_season_prem_matches = attach_match_elo(this_season_prem_matches, elo_long)

played, team_fixtures = build_team_table(this_season_prem_matches, GW)
prev_played, prev_team_fixtures = build_team_table(prev_fx, 38)

played      = add_diffs(played, team_fixtures)
prev_played = add_diffs(prev_played, prev_team_fixtures)

latest = print_team_strength_diagnostics(team_fixtures, teams, this_season_prem_matches)

Empty DataFrame
Columns: [date, gw, code, elo, estimated]
Index: []
short_name  weighted_attack
       MCI             1.05
       BOU             1.03
       BHA             0.85
       MUN             0.82
       ARS             0.77
       LIV             0.73
       FUL             0.71
       CHE             0.71
       NFO             0.69
       TOT             0.68
       EVE             0.67
       AVL             0.66
       BRE             0.65
       IPS             0.63
       NEW             0.62
       CRY             0.62
       LEE             0.60
       SUN             0.58
       COV             0.51
       HUL             0.43
short_name  weighted_defence
       ARS             -0.34
       BOU             -0.50
       MCI             -0.53
       TOT             -0.55
       BHA             -0.57
       LEE             -0.60
       NFO             -0.60
       FUL             -0.61
       NEW             -0.64
       LIV             -0.67
       CHE             -0

In [10]:
train, test, FEATURES = build_train_test(played, prev_played, GW)

# ---- exploratory: Poisson vs flat baseline on this-season holdout ----
tr = train[train["season"] == 0]
ho = train[train["season"] == 1]

def fit_pred(y_col):
    m = sm.GLM(tr[y_col], sm.add_constant(tr[FEATURES]),
               family=sm.families.Poisson()).fit()
    p = m.predict(sm.add_constant(ho[FEATURES], has_constant="add"))
    return m, p

m_h, p_h = fit_pred("home_non_penalty_xg")
m_a, p_a = fit_pred("away_non_penalty_xg")

flat_h = tr["home_non_penalty_xg"].mean()
flat_a = tr["away_non_penalty_xg"].mean()

print("home  poisson", mean_poisson_deviance(ho["home_non_penalty_xg"], p_h),
      "flat", mean_poisson_deviance(ho["home_non_penalty_xg"], [flat_h]*len(ho)))
print("away  poisson", mean_poisson_deviance(ho["away_non_penalty_xg"], p_a),
      "flat", mean_poisson_deviance(ho["away_non_penalty_xg"], [flat_a]*len(ho)))
print(m_h.summary().tables[1])

last season home_days_rest 70 of 380
last season away_days_rest 68 of 380
this season home_days_rest 26 of 110
this season away_days_rest 19 of 110
True True
(480, 138) season
0    380
1    100
dtype: int64
home  poisson 0.3877769808649139 flat 0.4459834829176172
away  poisson 0.37482786805946594 flat 0.3763386229510635
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.3490      0.043      8.040      0.000       0.264       0.434
elo_context     0.3155      0.076      4.148      0.000       0.166       0.465


In [11]:
# ---- exploratory: pooled model with is_home ----
def stack_home_and_away(match_frame):
    home_rows = pd.DataFrame({
        "team_xg": match_frame["home_non_penalty_xg"],
        "elo_context": match_frame["elo_context"],
        "is_home": 1,
    })
    away_rows = pd.DataFrame({
        "team_xg": match_frame["away_non_penalty_xg"],
        "elo_context": -match_frame["elo_context"],   # away team's perspective
        "is_home": 0,
    })
    return pd.concat([home_rows, away_rows], ignore_index=True)

stacked_last_season = stack_home_and_away(tr)
stacked_this_season = stack_home_and_away(ho)
print(stacked_last_season.groupby("is_home")["team_xg"].mean())
print(stacked_this_season.groupby("is_home")["team_xg"].mean())

pooled_features = ["elo_context", "is_home"]

pooled_model = sm.GLM(stacked_last_season["team_xg"],
                      sm.add_constant(stacked_last_season[pooled_features]),
                      family=sm.families.Poisson()).fit()

print(pooled_model.summary().tables[1])
print("learned home advantage:", np.exp(pooled_model.params["is_home"]))

is_home
0    1.156105
1    1.439789
Name: team_xg, dtype: float64
is_home
0    1.2887
1    1.4684
Name: team_xg, dtype: float64
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.1265      0.048      2.631      0.009       0.032       0.221
elo_context     0.3388      0.057      5.982      0.000       0.228       0.450
is_home         0.2201      0.064      3.436      0.001       0.095       0.346
learned home advantage: 1.246186638800305


In [15]:
pooled_predictions = pooled_model.predict(
    sm.add_constant(stacked_this_season[pooled_features], has_constant="add"))

for side_name, home_flag, separate_predictions, flat_value in [
        ("home", 1, p_h, flat_h), ("away", 0, p_a, flat_a)]:
    side_mask = stacked_this_season["is_home"] == home_flag
    actual_xg = stacked_this_season.loc[side_mask, "team_xg"]
    print(side_name,
          "| pooled:", round(mean_poisson_deviance(actual_xg, pooled_predictions[side_mask]), 3),
          "| separate:", round(mean_poisson_deviance(actual_xg, separate_predictions), 3),
          "| flat:", round(mean_poisson_deviance(actual_xg, [flat_value] * len(actual_xg)), 3))

home | pooled: 0.388 | separate: 0.388 | flat: 0.446
away | pooled: 0.371 | separate: 0.375 | flat: 0.376


In [18]:
#### POISSON REGRESSION
pooled_model = fit_pooled_goal_model(train, FEATURES)
test, predictions = predict_fixtures(test, train, pooled_model, FEATURES, teams,
                                     output_path="predicted_fixtures_GW.csv")

## for walk forward
# train = train.sort_values(["season", "gameweek"])

pooled home advantage: 1.222
             elo_context
elo_context          1.0
  home_team_name away_team_name  open_play_diff  set_play_diff  \
0            MCI            FUL        0.770102      -0.116011   

   short_rest_advantage  
0                     0  


In [19]:
train["home_set"]  = train["home_weighted_set_play_strength"]  - train["away_weighted_set_play_defence"]
train["home_open"] = train["home_weighted_open_play_strength"] - train["away_weighted_open_play_defence"]
train["away_set"]  = train["away_weighted_set_play_strength"]  - train["home_weighted_set_play_defence"]
train["away_open"] = train["away_weighted_open_play_strength"] - train["home_weighted_open_play_defence"]

experiment_train = train.dropna(subset=["home_set", "home_open", "away_set", "away_open"])
tr = experiment_train[experiment_train["season"] == 0]
ho = experiment_train[experiment_train["season"] == 1]

def score(y_col, feats):
    m = sm.GLM(tr[y_col], sm.add_constant(tr[feats]), family=sm.families.Poisson()).fit()
    p = m.predict(sm.add_constant(ho[feats], has_constant="add"))
    return round(mean_poisson_deviance(ho[y_col], p), 4), m

for side_name, y, s, o in [("home", "home_non_penalty_xg", "home_set", "home_open"),
                           ("away", "away_non_penalty_xg", "away_set", "away_open")]:
    print(side_name,
          "elo-only", score(y, ["elo_context"])[0],
          "| set+elo", score(y, [s, "elo_context"])[0],
          "| set+open+elo", score(y, [s, o, "elo_context"])[0])

print("away xG mean  last season:", tr["away_non_penalty_xg"].mean(),
      "| this season:", ho["away_non_penalty_xg"].mean())
print("home xG mean  last season:", tr["home_non_penalty_xg"].mean(),
      "| this season:", ho["home_non_penalty_xg"].mean())

home elo-only 0.3689 | set+elo 0.3688 | set+open+elo 0.3738
away elo-only 0.3657 | set+elo 0.3664 | set+open+elo 0.3634
away xG mean  last season: 1.1567297297297296 | this season: 1.3090000000000002
home xG mean  last season: 1.4435135135135135 | this season: 1.426111111111111


In [20]:
# ---- FIXTURE SCORE from predictor (single GW) ----
team_fx = build_team_fixture_scores(test)
print(merged["player_id"].duplicated().sum())

count    20.000000
mean      0.278923
std       0.082023
min       0.158366
25%       0.203834
50%       0.276300
75%       0.353272
max       0.407379
Name: cs_prob, dtype: float64
0    0.204354
1    0.209795
2    0.330053
3    0.224717
4    0.366844
5    0.393267
6    0.346551
7    0.265823
8    0.233794
9    0.352682
Name: cs_prob, dtype: float64
0


In [21]:
merged = attach_team_and_fixture_scores(merged, team_fixtures, team_fx)

['id', 'status', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'now_cost', 'now_cost_rank', 'now_cost_rank_type', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'selected_by_percent', 'selected_rank', 'selected_rank_type', 'total_points', 'event_points', 'points_per_game', 'points_per_game_rank', 'points_per_game_rank_type', 'bonus', 'bps', 'form', 'form_rank', 'form_rank_type', 'value_form', 'value_season', 'dreamteam_count', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'ep_next', 'ep_this', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'expected_goals_per_90', 'expected_assists_per_90', 'expected_goal_involvements_per_90', 'expected_goals_conceded_per_90', 'influence', 'influence_rank', 'influence_rank_type', 'creativity', 'creativity_rank', 'creativity_rank_type', 'threat', 'threat_rank', 'threat_rank_type', 'ict_index', 'ict_index_rank',

In [22]:
###LAST SEASON'S STATS
merged = attach_last_season_stats(merged, last_year_stats)

In [23]:
merged = attach_minutes_factors(merged, LAST_GW)

['id', 'status', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'now_cost', 'now_cost_rank', 'now_cost_rank_type', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'selected_by_percent', 'selected_rank', 'selected_rank_type', 'total_points', 'event_points', 'points_per_game', 'points_per_game_rank', 'points_per_game_rank_type', 'bonus', 'bps', 'form', 'form_rank', 'form_rank_type', 'value_form', 'value_season', 'dreamteam_count', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'ep_next', 'ep_this', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'expected_goals_per_90', 'expected_assists_per_90', 'expected_goal_involvements_per_90', 'expected_goals_conceded_per_90', 'influence', 'influence_rank', 'influence_rank_type', 'creativity', 'creativity_rank', 'creativity_rank_type', 'threat', 'threat_rank', 'threat_rank_type', 'ict_index', 'ict_index_rank',

In [24]:
merged = attach_expected_return_signals(merged, LAST_GW)

count    410.000000
mean       0.389295
std        0.446084
min       -0.952652
25%       -0.023497
50%        0.496528
75%        0.786083
max        1.000000
Name: fixture_score, dtype: float64
   web_name   position  minutes  current_x_fpl_p90  current_x_fpl_p90_scaled
Hinshelwood Midfielder      157           8.023522                  3.491167
    Gabriel   Defender      900           7.200000                  2.700670
  De Cuyper   Defender      822           7.154427                  2.656924
       Saka Midfielder      596           6.985820                  2.495079
   Vuskovic   Defender      887           6.819336                  2.335270
     Araujo   Defender      663           6.725133                  2.244846
      Foden Midfielder      361           6.558807                  2.085190
  Calafiori   Defender      767           6.528657                  2.056248
   Gelhardt Midfielder      128           6.399769                  1.932529
  Tavernier Midfielder      840   

In [25]:
board = assemble_board(merged)

Goalkeeper 24 -0.46081692700877985
Defender 145 0.27284945165133717
Midfielder 196 0.31328429723574297
Forward 45 -0.39546348108468576
Goalkeeper 24 -0.4747156039422886
Defender 145 0.3610014345376867
Midfielder 196 0.4795464472112054
Forward 45 -0.10551474987487258


In [ ]:
all_boards = build_all_boards(config.LEAGUES)

players_to_watch = {557, 565, 572, 579, 605, 608}

for league_board in build_all_boards():
    print(league_board[league_board["player_id"].isin(players_to_watch)].to_string(index=False))
    print()

dict_keys(['name', 'entry_id', 'league_id', 'league_size', 'format', 'known_taken', 'known_mine', 'known_available'])
Gegen
 player_id  is_mine  is_available  is_taken
       572    False         False      True
       565    False         False      True
       579    False         False      True
       605    False          True     False
       608    False          True     False
       557     True         False      True
dict_keys(['name', 'entry_id', 'league_id', 'league_size', 'format', 'known_taken', 'known_mine', 'known_available'])
Dalot
 player_id  is_mine  is_available  is_taken
       557    False         False      True
       565    False         False      True
       572    False         False      True
       579    False          True     False
       605    False         False      True
       608    False          True     False


In [ ]:
enriched = enrich_and_export(board, all_boards, LAST_GW, output_path="weekly_board_enriched.csv")

print_player_debug(merged)

print

['id', 'status', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'now_cost', 'now_cost_rank', 'now_cost_rank_type', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'selected_by_percent', 'selected_rank', 'selected_rank_type', 'total_points', 'event_points', 'points_per_game', 'points_per_game_rank', 'points_per_game_rank_type', 'bonus', 'bps', 'form', 'form_rank', 'form_rank_type', 'value_form', 'value_season', 'dreamteam_count', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'ep_next', 'ep_this', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'expected_goals_per_90', 'expected_assists_per_90', 'xgi_p90', 'expected_goals_conceded_per_90', 'influence', 'influence_rank', 'influence_rank_type', 'creativity', 'creativity_rank', 'creativity_rank_type', 'threat', 'threat_rank', 'threat_rank_type', 'ict_index', 'ict_index_rank', 'ict_index_rank_type', 'c

<function print(*args, sep=' ', end='\n', file=None, flush=False)>